# Galaxy Merger Tracking — SIMBA cis100

Runs the full merger-detection pipeline on real SIMBA-100 data.

**Prerequisites**
- `simbanator` registered for `cis100` in `~/.simbanator/config.json`
- Progenitor track file at `output/cis100/progenitors/progenitors_most_mass.fits`  
  (produced by `caesar_read_progen` — see `test_sfh_caesar.ipynb`)

**Pipeline**
1. Load simulation handle and progenitor track file  
2. Run `process_galaxies_with_tracks` — find companions in each snapshot  
3. Run `analyze_mergers` — classify major / minor events  
4. Visualise results

## 1 · Setup

In [ ]:
import sys, os
_pkg_root = '/home/lorenzong/analize_simba_cgm'
if _pkg_root not in sys.path:
    sys.path.insert(0, _pkg_root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from simbanator.io.simba import Simulation
from simbanator.io.paths import OutputPaths
from simbanator.analysis.mergers import (
    process_galaxies_with_tracks,
    analyze_mergers,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── merger-detection parameters ──────────────────────────────────────────
BOX_SIZE             = 100.0   # Mpc/h — SIMBA 100 box
SEARCH_RADIUS_FACTOR = 5.0     # search sphere = factor × r_half
MASS_THRESHOLD       = 1e9     # min companion stellar mass (M_sun)
RHALF_UNIT_FACTOR    = 1e-3    # kpc/h → Mpc/h (Caesar default units)

MASS_RATIO_MAJOR     = 0.25    # >= major merger
MASS_RATIO_MINOR     = 0.10    # >= minor merger (< major)

## 2 · Simulation handle and track file

In [ ]:
sim = Simulation('cis100')
out = OutputPaths(sim.name)

track_path = os.path.join(out.progenitors, 'progenitors_most_mass.fits')
print('Track file :', track_path)
print('Exists     :', os.path.exists(track_path))

In [ ]:
# Inspect the track file structure
with fits.open(track_path) as hdul:
    hdul.info()
    colnames = list(hdul[1].columns.names)
    n_rows   = len(hdul[1].data)

# Column names are snapshot numbers (plus 'GroupID' written by caesar_read_progen)
snap_cols = [c for c in colnames if c.upper() != 'GROUPID']
snaplist  = sorted([int(c) for c in snap_cols])   # ascending: 44, 45, … 150

print(f'\nTracked galaxies : {n_rows}')
print(f'Snapshot columns : {len(snap_cols)}')
print(f'Snapshot range   : {snaplist[0]} → {snaplist[-1]}')
print(f'All columns      : {colnames[:4]} … {colnames[-3:]}')

### Snapshot → redshift mapping
Loaded from the bundled scale-factor table (no live catalog access needed).

In [ ]:
# Load the bundled snap→redshift table for SIMBA-100
_data_dir = os.path.join(_pkg_root, 'simbanator', 'data', 'snap_z_maps')
scale_factors = np.loadtxt(os.path.join(_data_dir, 'zsnap_map_caesar_box100.txt'))

def snap_to_z(s):
    return 1.0 / scale_factors[int(s)] - 1.0

redshifts = np.array([snap_to_z(s) for s in snaplist])
print(f'z range: {redshifts.min():.3f} (snap {snaplist[0]}) → '
      f'{redshifts.max():.3f} (snap {snaplist[-1]})')

## 3 · Find merger companions

`process_galaxies_with_tracks` reads the Caesar HDF5 catalogs directly and records all
neighbours within `SEARCH_RADIUS_FACTOR × r_half` as `Progenitor` entries.

> **Note on track file format**  
> `caesar_read_progen` writes a `GroupID` column plus one column per snapshot  
> (named by snapshot number, e.g. `'44'`, `'45'`, …).  
> The updated `process_galaxies_with_tracks` now handles this automatically —  
> it drops `GroupID` and sorts columns by snapshot number.

In [ ]:
# Build catalog paths in the same ascending order as the track file columns
caesar_paths = [sim.get_caesar_file(s) for s in snaplist]

galaxies = process_galaxies_with_tracks(
    track_fits_path      = track_path,
    box_size             = BOX_SIZE,
    caesar_paths         = caesar_paths,
    search_radius_factor = SEARCH_RADIUS_FACTOR,
    mass_threshold       = MASS_THRESHOLD,
    rhalf_unit_factor    = RHALF_UNIT_FACTOR,
)

In [ ]:
total_progs = sum(len(g.progenitors) for g in galaxies.values())
gals_with_progs = sum(1 for g in galaxies.values() if g.progenitors)

print(f'Tracked galaxies          : {len(galaxies)}')
print(f'Galaxies with companions  : {gals_with_progs}')
print(f'Total companion entries   : {total_progs}')

### Diagnostic — search radius sanity check
Verify that the search radii are physically reasonable before trusting the companion counts.

In [ ]:
# Sample the search radius from the Galaxy objects (set at snap_idx=0, i.e. snap 44)
r_halfs = np.array([g.r for g in galaxies.values() if g.r > 0])
if len(r_halfs):
    r_search_kpc = r_halfs * SEARCH_RADIUS_FACTOR        # kpc/h (before unit conversion)
    r_search_mpc = r_search_kpc * RHALF_UNIT_FACTOR      # Mpc/h
    print(f'r_half range (kpc/h)        : {r_halfs.min():.1f} – {r_halfs.max():.1f}')
    print(f'Search radius range (kpc/h) : {r_search_kpc.min():.1f} – {r_search_kpc.max():.1f}')
    print(f'Search radius range (Mpc/h) : {r_search_mpc.min():.4f} – {r_search_mpc.max():.4f}')
else:
    print('No r_half values stored — galaxies were absent at snap 44 (first column).')

## 4 · Classify mergers

In [ ]:
n_snaps = len(snaplist)
n_gals  = len(galaxies)

major_mergers, minor_mergers = analyze_mergers(
    galaxies,
    array_size         = (n_snaps, n_gals),
    mass_threshold_maj = MASS_RATIO_MAJOR,
    mass_threshold_min = MASS_RATIO_MINOR,
)

print('Array shape (n_snaps, n_gals):', major_mergers.shape)
print(f'Total major merger events : {major_mergers.sum()}')
print(f'Total minor merger events : {minor_mergers.sum()}')

In [ ]:
# Per-galaxy merger counts (top 20 by total events)
total_per_gal = major_mergers.sum(axis=0) + minor_mergers.sum(axis=0)
top_idx = np.argsort(total_per_gal)[::-1][:20]

print(f'{"Galaxy idx":>12}  {"Major":>8}  {"Minor":>8}  {"Total":>8}')
print('-' * 44)
for idx in top_idx:
    maj = int(major_mergers[:, idx].sum())
    mni = int(minor_mergers[:, idx].sum())
    if maj + mni == 0:
        break
    print(f'{idx:>12}  {maj:>8}  {mni:>8}  {maj+mni:>8}')

## 5 · Visualisation

In [ ]:
# Merger rate vs redshift (sample-total per snapshot)
fig, ax = plt.subplots(figsize=(9, 4))

ax.step(redshifts, major_mergers.sum(axis=1), where='mid',
        color='firebrick', lw=2, label='Major (ratio ≥ 0.25)')
ax.step(redshifts, minor_mergers.sum(axis=1), where='mid',
        color='steelblue', lw=2, label='Minor (0.10 ≤ ratio < 0.25)')

ax.set_xlabel('Redshift z')
ax.set_ylabel('Number of merger events')
ax.set_title('Merger count per snapshot (all tracked galaxies)')
ax.invert_xaxis()
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-galaxy merger history heatmap (only galaxies that have at least one event)
active = np.where((major_mergers.sum(axis=0) + minor_mergers.sum(axis=0)) > 0)[0]

if len(active) == 0:
    print('No merger events found — nothing to plot.')
else:
    combined = np.zeros((n_snaps, len(active)), dtype=int)
    combined[minor_mergers[:, active] > 0] = 1
    combined[major_mergers[:, active] > 0] = 2

    fig, ax = plt.subplots(figsize=(11, max(3, len(active) * 0.25 + 1)))
    cmap = plt.cm.get_cmap('RdYlGn_r', 3)
    im = ax.imshow(combined.T, aspect='auto', origin='lower',
                   cmap=cmap, vmin=-0.5, vmax=2.5,
                   extent=[redshifts[0], redshifts[-1], -0.5, len(active) - 0.5])

    cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2], pad=0.02)
    cbar.ax.set_yticklabels(['None', 'Minor', 'Major'])

    ax.set_xlabel('Redshift z')
    ax.set_ylabel('Galaxy index (active only)')
    ax.set_title(f'Merger history — {len(active)} galaxies with at least one event')
    ax.set_yticks(range(len(active)))
    ax.set_yticklabels(active)
    plt.tight_layout()
    plt.show()

In [ ]:
# Mass-ratio distribution of all recorded companions
all_mass_ratios = np.concatenate([
    gal.get_attribute_values('mass')
    for gal in galaxies.values()
    if gal.progenitors
])

if len(all_mass_ratios) == 0:
    print('No companions recorded — skipping mass-ratio plot.')
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = np.linspace(0, 1, 30)
    ax.hist(all_mass_ratios, bins=bins, color='slategray', edgecolor='white', lw=0.4)
    ax.axvline(MASS_RATIO_MAJOR, color='firebrick', ls='--', lw=1.5,
               label=f'Major threshold ({MASS_RATIO_MAJOR})')
    ax.axvline(MASS_RATIO_MINOR, color='steelblue', ls='--', lw=1.5,
               label=f'Minor threshold ({MASS_RATIO_MINOR})')
    ax.set_xlabel(r'Stellar mass ratio  $m_{\rm sec}\,/\,m_{\rm main}$')
    ax.set_ylabel('Count')
    ax.set_title('Mass-ratio distribution of merger companions')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Companion separation distribution
all_distances = np.concatenate([
    gal.get_attribute_values('distance')
    for gal in galaxies.values()
    if gal.progenitors
])

if len(all_distances) == 0:
    print('No companions recorded — skipping distance plot.')
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(all_distances, bins=40, color='mediumpurple', edgecolor='white', lw=0.4)
    ax.set_xlabel('Separation [Mpc/h]')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of companion separations')
    plt.tight_layout()
    plt.show()

## 6 · Save results

In [ ]:
import h5py

save_dir  = out.subdir('mergers')
save_path = os.path.join(save_dir, 'merger_arrays.hdf5')

with h5py.File(save_path, 'w') as f:
    f.create_dataset('major_mergers', data=major_mergers)
    f.create_dataset('minor_mergers', data=minor_mergers)
    f.create_dataset('snaplist',      data=np.array(snaplist))
    f.create_dataset('redshifts',     data=redshifts)
    f.attrs['n_galaxies']            = n_gals
    f.attrs['search_radius_factor']  = SEARCH_RADIUS_FACTOR
    f.attrs['mass_threshold_msun']   = MASS_THRESHOLD
    f.attrs['mass_ratio_major']      = MASS_RATIO_MAJOR
    f.attrs['mass_ratio_minor']      = MASS_RATIO_MINOR

print('Saved to:', save_path)